# Reviews de produtos (pt-BR): exploratory analysis

2.87 million Brazilian product reviews, the clauses cut from them and the pros and cons computed in 2020. On Kaggle the notebook reads the full Parquet files; in the repository it reads the sample (2,000 products). Documentation: [github.com/CaiqueDourado/reviews-produtos-ptbr](https://github.com/CaiqueDourado/reviews-produtos-ptbr).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)
full = [p.parent for p in Path('/kaggle/input').glob('**/reviews.parquet')]
if full:
    d = full[0]
    reviews = pd.read_parquet(d / 'reviews.parquet', columns=['review_id', 'group_id', 'store', 'rating', 'recommended',
                                                              'title', 'text', 'tilde_restored', 'sentiment', 'helpfulness', 'submitted_at'])
    pros_cons = pd.read_parquet(d / 'pros_cons.parquet')
    aspects = pd.read_parquet(d / 'aspect_ratings.parquet')
    groups = pd.read_parquet(d / 'groups.parquet')
    sentences_path = d / 'sentences.parquet'
else:
    d = next(p for p in (Path.cwd() / 'sample', Path.cwd().parent / 'sample') if p.exists())
    reviews = pd.read_csv(d / 'reviews_sample.csv', parse_dates=['submitted_at'])
    pros_cons = pd.read_csv(d / 'pros_cons_sample.csv')
    aspects = pd.read_csv(d / 'aspect_ratings_sample.csv')
    groups = pd.read_csv(d / 'groups_sample.csv')
    sentences_path = d / 'sentences_sample.csv'
print(f"{len(reviews):,} reviews, {reviews.group_id.nunique():,} products")

## Stars and stores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
reviews['rating'].value_counts().sort_index().plot.bar(ax=axes[0], color='#2a78d6', title='Reviews per star')
reviews.groupby('store')['rating'].mean().sort_values().plot.barh(ax=axes[1], color='#1f9e89', title='Average stars per store')
axes[1].set_xlim(3, 5); axes[0].set_xlabel(''); axes[1].set_ylabel('')
plt.tight_layout()

## Reviews per year

In [ ]:
reviews['submitted_at'].dt.year.value_counts().sort_index().plot.bar(figsize=(9, 3), color='#2a78d6', title='Reviews by year written')
plt.xlabel(''); plt.tight_layout()

## AWS sentiment against the stars

The label is for the whole review. POSITIVE reviews with 1-2 stars were relabelled NEGATIVE in 2020, so those cells are empty.

In [ ]:
pd.crosstab(reviews['rating'], reviews['sentiment'], normalize='index').round(3).mul(100)

## Pros and cons of the products with most reviews

In [ ]:
top = groups.sort_values('reviews', ascending=False).head(5)
for g in top.itertuples():
    pc = pros_cons[pros_cons.group_id == g.group_id]
    print(f"\n{g.group_name}  ({g.reviews} reviews, {g.avg_rating} stars)")
    for kind, label in (('pro', '+'), ('con', '-')):
        print(f"  {label} " + ' | '.join(pc[pc.kind == kind].sort_values('rank').clause))

## Aspect scores

The review forms asked for scores per aspect. Which aspects pull the average down?

In [ ]:
aspects.groupby('aspect')['score'].agg(['count', 'mean']).query('count > 50').sort_values('mean').round(2)

## Clauses: the most common ones

`count_site` is how often a clause appears in the whole database.

In [ ]:
cols = ['text', 'canonical', 'count_site']
sentences = (pd.read_parquet(sentences_path, columns=cols) if str(sentences_path).endswith('.parquet')
             else pd.read_csv(sentences_path, usecols=cols))
sentences.drop_duplicates('text').nlargest(15, 'count_site')

## Restored ã/õ

Texts that lost every ã/õ in collection were fixed from the spelling seen in intact texts.

In [ ]:
print(f"{reviews['tilde_restored'].mean():.1%} of the reviews had ã/õ restored")
reviews.loc[reviews['tilde_restored'], ['text']].head(5)